In [4]:
from hana_ml import dataframe
url, port, user, passwd = "810070ba-df1a-4553-9a82-23690dc0158e.hana.demo-hc-3-haas-hc-dev.dev-aws.hanacloud.ondemand.com", \
443, "SAPUSER", "nIL0yr8S"
#url='hcp-ml-validate.hana-ml.c.ap-cn-1.cloud.sap'
#user='MLAPITESTER'
#passwd='Abcd12345'
#port=30315
cc = dataframe.ConnectionContext(url, port, user, passwd)

In [5]:
import numpy as np
import pandas as pd
np.random.seed(2023)
seq_len = 16
data = pd.concat((pd.DataFrame(dict(DATE=pd.date_range('2025-01-01', periods=seq_len),
                                    ID=range(seq_len))),
                  pd.DataFrame(10 * np.random.normal(size=(seq_len, 2)), columns=['X1', 'X2'])),
                  axis=1)

In [6]:
from hana_ml.dataframe import create_dataframe_from_pandas
sim_df = create_dataframe_from_pandas(cc, data,
                                      'SIM_DATA_TBL',
                                      force=True)

100%|██████████| 1/1 [00:00<00:00,  2.62it/s]


In [7]:
from hana_ai.tools.hana_ml_tools.correlation_tools import Correlation
cf_tool = Correlation(cc)

In [8]:
from langchain.agents import initialize_agent, AgentType
from gen_ai_hub.proxy.langchain import init_llm
llm = init_llm('gpt-4o', temperature=0.0, max_tokens=2000) # used to do logical reasoning
tools = [cf_tool] # Add any tools here
agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)

C:\Users\I326292\AppData\Local\Temp\ipykernel_9376\2058210661.py:5: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)


### Test 1 : Timestamp Index

In [5]:
tool_input = dict(table_name='SIM_DATA_TBL',
                  key='DATE',
                  x='X3',
                  calculate_pacf=True)
                  #y='X2',
                  #max_lag=10)
cf_tool.run(tool_input=tool_input)

'{"Error message": "Column(s) not in DataFrame: [\'X3\']"}'

In [16]:
cc.table("SIM_DATA_TBL_CORRELATION_RESULT").head(3).collect()

,LAG,CV,CF,PACF
0,-4,-6.643336,-0.058390,None
1,-3,8.214894,0.072203,None
2,-2,2.859731,0.025135,None


In [6]:
instruction = "Please compute the autocorrelation of the time-series data in column X3 of table " +\
"SIM_DATA_TBL along with its partial auto-correlation coefficients, where key is DATE"
agent_chain.invoke(instruction)

{'input': 'Please compute the autocorrelation of the time-series data in column X3 of table SIM_DATA_TBL along with its partial auto-correlation coefficients, where key is DATE',
 'output': "It seems there was an error because the column 'X3' does not exist in the table 'SIM_DATA_TBL'. Please verify the column name and provide the correct one, or check if the table name is correct."}

In [13]:
cc.table("SIM_DATA_TBL_CORRELATION_RESULT").collect()

,LAG,CV,CF,PACF
0,0,121.881690,1.000000,1.000000
1,1,29.159637,0.239245,0.239245
2,2,-14.649828,-0.120197,-0.188208
3,3,-12.993741,-0.106609,-0.030145
4,4,-5.505351,-0.045170,-0.032853


### Test 2 : ACFs with Confidence Interval

In [22]:
tool_input = dict(table_name='SIM_DATA_TBL',
                  key='ID',
                  x='X1',
                  calculate_confint=True)
                  #max_lag=10)
cf_tool.run(tool_input=tool_input)

'{"correlation_result_table": "SIM_DATA_TBL_CORRELATION_RESULT"}'

In [23]:
cc.table("SIM_DATA_TBL_CORRELATION_RESULT").collect()

,LAG,CV,CF,PACF,ACF_CONFIDENCE_BOUND,PACF_CONFIDENCE_BOUND
0,0,121.881690,1.000000,1.000000,NaN,NaN
1,1,29.159637,0.239245,0.239245,0.489991,0.489991
2,2,-14.649828,-0.120197,-0.188208,0.517278,0.489991
3,3,-12.993741,-0.106609,-0.030145,0.523940,0.489991
4,4,-5.505351,-0.045170,-0.032853,0.529123,0.489991


In [24]:
instruction = "Please compute the autocorrelation of the time-series data in column X1 of table " +\
"SIM_DATA_TBL along with its partial auto-correlation coefficients as well as confidence intervals, " +\
"where key is DATE"
agent_chain.invoke(instruction)

{'input': 'Please compute the autocorrelation of the time-series data in column X1 of table SIM_DATA_TBL along with its partial auto-correlation coefficients as well as confidence intervals, where key is DATE',
 'output': 'The autocorrelation, partial autocorrelation coefficients, and confidence intervals for the time-series data in column X1 of table SIM_DATA_TBL have been computed and stored in the table SIM_DATA_TBL_CORRELATION_RESULT.'}

In [25]:
cc.table("SIM_DATA_TBL_CORRELATION_RESULT").collect()

,LAG,CV,CF,PACF,ACF_CONFIDENCE_BOUND,PACF_CONFIDENCE_BOUND
0,0,121.881690,1.000000,1.000000,NaN,NaN
1,1,29.159637,0.239245,0.239245,0.489991,0.489991
2,2,-14.649828,-0.120197,-0.188208,0.517278,0.489991
3,3,-12.993741,-0.106609,-0.030145,0.523940,0.489991
4,4,-5.505351,-0.045170,-0.032853,0.529123,0.489991


### Case 3 : Correlation between Two Columns

In [ ]:
instruction = "Please compute the correlation between column X1 and column X2 in table SIM_DATA_TBL, "+\
", where key is ID and maximum lag is 3."
agent_chain.invoke(instruction)

{'input': 'Please compute the correlation between column X1 and column X2 in table SIM_DATA_TBL, inclusive of the partial auto-correlation coefficients, where key is ID and maximum lag is 3.',
 'output': 'The correlation computation has been completed, and the results are stored in the table `SIM_DATA_TBL_CORRELATION_RESULT`. You can check this table for the correlation and partial auto-correlation coefficients between columns X1 and X2.'}

In [28]:
cc.table("SIM_DATA_TBL_CORRELATION_RESULT").collect()

,LAG,CV,CF,PACF
0,-3,8.214894,0.072203,None
1,-2,2.859731,0.025135,None
2,-1,19.379410,0.170331,None
3,0,36.325105,0.319272,None
4,1,-13.231161,-0.116292,None
5,2,-14.500138,-0.127446,None
6,3,18.568941,0.163208,None


### Test Case 4 : Misconfigurated Parameter Settings

In [9]:

tool_input = dict(table_name='SIM_DATA_TBL',
                    key='ID',
                    x='X1',
                    y="X2",
                    calculate_confint=True)
cf_tool.run(tool_input=tool_input)

'{"Error message": "confidence intervals are only applicable to the autocorrelation of one time-series."}'

In [19]:
instruction = "Please compute the auto-correlation of DATE in table SIM_DATA_TBL "+\
"together with the confidence intervals of the coefficients, where key is ID."
agent_chain.invoke(instruction)

ERROR:hana_ml.algorithms.pal.tsa.correlation_function:(423, 'AFL error: AFL DESCRIBE for nested call failed - invalid table(s) for ANY-procedure call (Input table 0: column 1 (starting with 0) has invalid SQL type.): line 9 col 1 (at pos 306)')
Traceback (most recent call last):
  File "c:\users\i326292\desktop\hanamlapi\src\hana_ml\algorithms\pal\tsa\correlation_function.py", line 173, in correlation
    sql, _ = call_pal_auto_with_hint(conn,
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\users\i326292\desktop\hanamlapi\src\hana_ml\algorithms\pal\pal_base.py", line 1416, in call_pal_auto_with_hint
    if try_exec(cur, sql, conn):
       ^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\users\i326292\desktop\hanamlapi\src\hana_ml\algorithms\pal\pal_base.py", line 1371, in try_exec
    cur.execute(sql)
hdbcli.dbapi.Error: (423, 'AFL error: AFL DESCRIBE for nested call failed - invalid table(s) for ANY-procedure call (Input table 0: column 1 (starting with 0) has invalid SQL type.): line 9 col 

{'input': 'Please compute the auto-correlation of DATE in table SIM_DATA_TBL together with the confidence intervals of the coefficients, where key is ID.',
 'output': 'The error message indicates that there is an issue with the data type of the DATE column in the SIM_DATA_TBL table. It seems that the DATE column might not be in a format that is suitable for auto-correlation computation. Typically, time-series data should be in a numeric format or a format that can be converted to numeric values.\n\nTo resolve this, we need to ensure that the DATE column is in a suitable format for analysis. If the DATE column is in a date or string format, it may need to be converted to a numeric format, such as a timestamp or an integer representing the number of days since a specific date.\n\nPlease check the format of the DATE column in the SIM_DATA_TBL table. If it is not in a numeric format, consider converting it to a suitable format before performing the auto-correlation computation. If you need

In [18]:
cc.table("SIM_DATA_TBL_CORRELATION_RESULT").collect()

,LAG,CV,CF,PACF,ACF_CONFIDENCE_BOUND,PACF_CONFIDENCE_BOUND
0,0,121.881690,1.000000,1.000000,NaN,NaN
1,1,29.159637,0.239245,0.239245,0.489991,0.489991
2,2,-14.649828,-0.120197,-0.188208,0.517278,0.489991
3,3,-12.993741,-0.106609,-0.030145,0.523940,0.489991
4,4,-5.505351,-0.045170,-0.032853,0.529123,0.489991


In [11]:
cc.drop_table("SIM_DATA_TBL_CORRELATION_RESULT")
cc.drop_table("SIM_DATA_TBL")

In [12]:
cc.close()